This notebook manually coerces raw Bauder ST AnnData to a standardized format for downstream analysis and concatenation.

Operation(s):

1. Manual curation of metadata
2. Mapping dimensions to reference genes
3. Addition of regional annotations
4. Basic QC filtering

Output(s):
1. DATADIR/7_spatial/harmonized/SP2-03.h5ad

Requirement(s):
1. DATADIR/7_spatial/harmonized/Bauder_spatial.h5ad
2. DATADIR/7_spatial/abc/25um_annotation_mtx.nrrd
3. DATADIR/7_spatial/structure_tree_processed.json
4. DATADIR/7_spatial/gene_metadata.csv

In the next cell you can indicate whether you want to skip the QC part of the notebook, where cells with less than 100 counts in HypoMap3D's HVGs are removed. 

If your goal is to only recreate the atlas, use SKIP_QC = False. 

However, if you want to recreate all HypoMap3D figures from the publication you will need all cells, in which case you use SKIP_QC = True.

In [1]:
SKIP_QC = True

In [2]:
import warnings
import json

from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import nrrd

from dotenv import dotenv_values
from scipy.sparse import csr_matrix

from hm3d_utils.utils import write_clean_h5ad
from project_utils import FileTracker

warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "7_spatial"
(OUTDIR / "harmonized").mkdir(exist_ok=True, parents=True) 

tracker = FileTracker(log_file=DATADIR/'tracked_files.toml')

infile = OUTDIR / "raw" / "Bauder_spatial.h5ad"

if SKIP_QC:
    outfile = OUTDIR / 'harmonized' / 'SP2-03_no_qc.h5ad'
else:
    outfile = OUTDIR / 'harmonized' / 'SP2-03.h5ad'

In [4]:
adata = sc.read_h5ad(infile)
adata

AnnData object with n_obs × n_vars = 600184 × 46995
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'cell_bc', 'x_pos', 'y_pos', 'cell_ID_mask', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_reads', 'reads_per_counts', 'n_joined', 'exact_entropy', 'theoretical_entropy', 'exact_compression', 'theoretical_compression', 'x_rot', 'y_rot', 'Cell_ID', 'Batch_ID', 'index', 'rotation', 'ccf_x', 'ccf_y', 'ccf_z', 'ccf_id', 'atlas_id', 'acronym', 'name', 'st_level', 'color_hex_triplet', 'parent_structure_id', 'is_hypo', 'ident'
    layers: 'logcounts'

In [5]:
adata.obs.keys()

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'cell_bc', 'x_pos', 'y_pos',
       'cell_ID_mask', 'n_genes_by_counts', 'total_counts', 'total_counts_mt',
       'pct_counts_mt', 'n_reads', 'reads_per_counts', 'n_joined',
       'exact_entropy', 'theoretical_entropy', 'exact_compression',
       'theoretical_compression', 'x_rot', 'y_rot', 'Cell_ID', 'Batch_ID',
       'index', 'rotation', 'ccf_x', 'ccf_y', 'ccf_z', 'ccf_id', 'atlas_id',
       'acronym', 'name', 'st_level', 'color_hex_triplet',
       'parent_structure_id', 'is_hypo', 'ident'],
      dtype='object')

In [6]:
## Bauder specific fixes

# Convert adata.X from float64 CSC to float32 CSR
adata.X = adata.X.astype(np.float32).tocsr()

# Map ensembl v107 gene names to gene stable ids to be mapped across to reference genes
ensembl_df = pd.read_csv('resources/Ensembl_v107_stable_ids.csv')
ensembl_df['Gene name'] = ensembl_df['Gene name'].fillna(ensembl_df['Gene stable ID']) # some genes have no names
gene_id_map = ensembl_df.set_index('Gene name')['Gene stable ID'].to_dict()

assert np.all(adata.var_names.isin(gene_id_map)), 'Not all genes match the provided reference assembly.'

adata.var = adata.var.rename(index=gene_id_map)

### Intialise new AnnData object

In [7]:
# Step 1: load all genes from reference from gene_metadata
gene_df = pd.read_csv(DATADIR / 'gene_metadata.csv', index_col=0)
ref_gene_ids = gene_df['gene_ids'].tolist()

# Step 2: Subset to genes only in reference
valid_genes_mask = np.asarray(adata.var_names.isin(ref_gene_ids))
print(f'{np.sum(~valid_genes_mask)} genes not in reference: {adata.var_names[~valid_genes_mask].tolist()}')
adata = adata[:, valid_genes_mask].copy()

# Step 2: get index of spatial genes in reference gene list
idx_map = np.array([ref_gene_ids.index(g) for g in adata.var_names])

# Step 3: construct reformatted AnnData
adata_harmonized = sc.AnnData(
    X=csr_matrix(
        (
            adata.X.data,
            idx_map[adata.X.indices],
            adata.X.indptr
        ),
        shape=(adata.shape[0], len(ref_gene_ids))
    ),
    var=gene_df[['gene_ids']].copy()
)

100 genes not in reference: ['ENSMUSG00000094958', 'ENSMUSG00000074473', 'ENSMUSG00000090863', 'ENSMUSG00000096923', 'ENSMUSG00000045506', 'ENSMUSG00000053706', 'ENSMUSG00000074210', 'ENSMUSG00000066378', 'ENSMUSG00000067292', 'ENSMUSG00000069518', 'ENSMUSG00000072693', 'ENSMUSG00000073290', 'ENSMUSG00000073682', 'ENSMUSG00000095891', 'ENSMUSG00000095186', 'ENSMUSG00000095547', 'ENSMUSG00000075014', 'ENSMUSG00000075015', 'ENSMUSG00000078091', 'ENSMUSG00000078488', 'ENSMUSG00000078620', 'ENSMUSG00000078912', 'ENSMUSG00000079010', 'ENSMUSG00000079264', 'ENSMUSG00000079286', 'ENSMUSG00000079353', 'ENSMUSG00000096385', 'ENSMUSG00000092157', 'ENSMUSG00000091604', 'ENSMUSG00000078197', 'ENSMUSG00000091441', 'ENSMUSG00000092004', 'ENSMUSG00000091159', 'ENSMUSG00000091306', 'ENSMUSG00000090441', 'ENSMUSG00000091041', 'ENSMUSG00000092345', 'ENSMUSG00000093574', 'ENSMUSG00000090625', 'ENSMUSG00000094296', 'ENSMUSG00000096240', 'ENSMUSG00000095316', 'ENSMUSG00000095346', 'ENSMUSG00000094204', 'EN

In [8]:
adata.obs.keys()

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'cell_bc', 'x_pos', 'y_pos',
       'cell_ID_mask', 'n_genes_by_counts', 'total_counts', 'total_counts_mt',
       'pct_counts_mt', 'n_reads', 'reads_per_counts', 'n_joined',
       'exact_entropy', 'theoretical_entropy', 'exact_compression',
       'theoretical_compression', 'x_rot', 'y_rot', 'Cell_ID', 'Batch_ID',
       'index', 'rotation', 'ccf_x', 'ccf_y', 'ccf_z', 'ccf_id', 'atlas_id',
       'acronym', 'name', 'st_level', 'color_hex_triplet',
       'parent_structure_id', 'is_hypo', 'ident'],
      dtype='object')

### Adding metadata to .obs

In [9]:
# Manually set dataset ID
adata_harmonized.obs['Dataset'] = 'SP2-03'
adata_harmonized.obs['Dataset'] = adata_harmonized.obs['Dataset'].astype('category')

# Map sample metadata to Sample
adata_harmonized.obs['Sample'] = pd.Categorical(adata.obs['Batch_ID'].values)

# Manually set Author
adata_harmonized.obs['Author'] = 'Bauder'
adata_harmonized.obs['Author'] = adata_harmonized.obs['Author'].astype('category')

# Manually set Technology
adata_harmonized.obs['Technology'] = 'Open-ST'
adata_harmonized.obs['Technology'] = adata_harmonized.obs['Technology'].astype('category')

# Map donor metadata to Donor
adata_harmonized.obs['Donor'] = adata.obs['Batch_ID'].str.split('_').str[:-1].str.join('_').values # extracted from sample ID
adata_harmonized.obs['Donor'] = pd.Categorical(
    adata_harmonized.obs['Donor']
)

# Map sex metadata to Sex (missing in Bauder)
adata_harmonized.obs['Sex'] = 'M'
adata_harmonized.obs['Sex'] = pd.Categorical(
    adata_harmonized.obs['Sex'],
    categories=["M", "F"]
)

# Set cell ids to dataset_sample_barcode
adata_harmonized.obs.index = (
    adata_harmonized.obs['Dataset'].astype(str) 
    + '_' 
    # + adata_harmonized.obs['Sample'].astype(str) # sample already in cell ids
    # + '_' 
    + adata.obs.index.astype(str)
).values

### Adding metadata to .var

In [10]:
# Rename 'gene_ids' to 'gene_stable_ids'
adata_harmonized.var = adata_harmonized.var.rename(columns={'gene_ids': 'gene_stable_ids'})
adata.var.index.name = 'gene_names'

# Add highly variable gene column
hvgs = pd.read_csv(DATADIR / '5_clustering' / 'clean_control_4500.csv', header=None, index_col=0).index.tolist()
adata_harmonized.var['IsHVG'] = adata_harmonized.var_names.isin(hvgs)

# Add a column indicating where a gene was captured by this technology
adata_harmonized.var['IsCaptured'] = adata_harmonized.var['gene_stable_ids'].isin(adata.var_names)

### Moving coordinates to .obsm

In [11]:
# We copy ccf coordinates as a reformatted dataframe to .obsm
ccf_df = adata.obs[['ccf_x', 'ccf_y', 'ccf_z']].copy()
ccf_df.index = adata_harmonized.obs.index
ccf_df.columns = ['x', 'y', 'z']
adata_harmonized.obsm['coords_ccf_mm'] = ccf_df

In [12]:
# We copy source coordinates as a reformatted dataframe to .obsm
source_df = adata.obs[['x_pos', 'y_pos']].copy()
source_df.index = adata_harmonized.obs.index
source_df.columns = ['x', 'y']
adata_harmonized.obsm['coords_source'] = source_df

### Adding regional annotations

In [13]:
# Load annotation voxel grid
annotation_nrrd_fp = OUTDIR / 'abc' / '25um_annotation_mtx.nrrd'
annotation_25um, metadata_25um = nrrd.read(annotation_nrrd_fp)
annotation_25um.shape

(528, 320, 456)

In [14]:
import numpy as np

# Convert coordinates
voxels = np.floor(
    adata_harmonized.obsm['coords_ccf_mm'] * 1000 / 25
).astype(int).values

x = voxels[:, 0]
y = voxels[:, 1]
z = voxels[:, 2]

# Volume shape
sx, sy, sz = annotation_25um.shape

# Build validity mask
valid = (
    (x >= 0) & (x < sx) &
    (y >= 0) & (y < sy) &
    (z >= 0) & (z < sz)
)

# Initialize output with 0 (your "out-of-bounds" label)
region_ids = np.zeros(voxels.shape[0], dtype=annotation_25um.dtype)

# Fill only valid positions
region_ids[valid] = annotation_25um[x[valid], y[valid], z[valid]]

adata_harmonized.obs['RegionID'] = region_ids

In [15]:
# Load structure tree
structure_tree_fp = OUTDIR / 'structure_tree_processed.json'
with open(structure_tree_fp, 'r') as fp:
    structure_tree = json.load(fp)

# Add 0 as an out of bounds (OOB) region to tree
structure_tree['0'] = {
    'acronym': 'OOB',
    'name': 'Out of Bound'
}

# Get region acronyms
id_acronym_map = {int(k): v['acronym'] for k, v in structure_tree.items()}
adata_harmonized.obs['RegionAcronym'] = adata_harmonized.obs['RegionID'].map(id_acronym_map).astype('category')

# Get region names
id_name_map = {int(k): v['name'] for k, v in structure_tree.items()}
adata_harmonized.obs['RegionName'] = adata_harmonized.obs['RegionID'].map(id_name_map).astype('category')

### Rough QC filtering

In [16]:
if SKIP_QC == False:

    MIN_HVG_COUNTS = 100

    adata_harmonized.obs['total_counts'] = np.asarray(adata_harmonized.X.sum(axis=1)).ravel()
    adata_harmonized.obs['total_hvg_counts'] = np.asarray(adata_harmonized[:, adata_harmonized.var['IsHVG']].X.sum(axis=1)).ravel()
    adata_harmonized.obs['total_genes'] = np.asarray((adata_harmonized.X > 0).sum(axis=1)).ravel()

    valid_mask = (adata_harmonized.obs['total_hvg_counts'] >= MIN_HVG_COUNTS) 

    print(f"{np.sum(~valid_mask)} cells removed with < {MIN_HVG_COUNTS} highly variable gene counts.")

    adata_harmonized = adata_harmonized[valid_mask]

### Save out new AnnData

In [17]:
tracker_descr = 'Harmonized Bauder Open-ST dataset.'

if SKIP_QC:
    tracker_descr += ' Not QC-filtered.'
else:
    tracker_descr += ' Filtered to cells with >= 100 counts in HypoMap3D HVGs.'

with tracker.register(outfile) as tf:
    tf.set_description(tracker_descr)
    write_clean_h5ad(adata_harmonized, tf.filepath)


In [18]:
adata_harmonized

AnnData object with n_obs × n_vars = 600184 × 56939
    obs: 'Dataset', 'Sample', 'Author', 'Technology', 'Donor', 'Sex', 'RegionID', 'RegionAcronym', 'RegionName'
    var: 'gene_stable_ids', 'IsHVG', 'IsCaptured'
    obsm: 'coords_ccf_mm', 'coords_source'